# RAG 检索层评估 Pipeline — 策略对比与面试展示

完整的 RAG 检索层离线评估流水线。加载 golden dataset，运行检索（支持真实/模拟两种模式），计算核心指标和上下文质量，可视化结果，并对比不同策略（embedding / chunking / top-k）的检索效果。

**适用场景：**
- 对比不同 chunking / embedding / retriever / top-k 策略
- 诊断 Recall vs. Precision 的权衡
- 识别噪声或重复过高的检索模式
- 导出 JSON 报告接入 CI 或 dashboard
- 加载多份实验报告进行策略横评
- 面试展示：一键生成对比图表与诊断结论

**使用流程：**
1. 修改 `.env` 中的配置（chunk_size / embedding_model 等）
2. 终端执行：`python scripts/clear_qdrant.py && rm -f data/ingestion_state.db`
3. 重新摄入：`python ingest.py --input_dir data/engineering --batch_size 64`
4. 运行评估：`python evaluation/run_retrieval_eval.py --dataset ... --experiment-name <策略名>`
5. 重复 1-4，更换不同配置
6. 回到本 notebook，`Restart & Run All` 即可看到完整对比

## 1. 环境与依赖

In [ ]:
from __future__ import annotations

import json
import os
import sys
import warnings
from pathlib import Path
from statistics import mean, stdev
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

# Mac 常见中文字体
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

warnings.filterwarnings("ignore", category=FutureWarning)

# ── 定位项目根目录：从当前目录向上查找包含 app/ 子目录的路径 ──
ROOT_DIR = Path.cwd()
while ROOT_DIR != ROOT_DIR.parent:
    if (ROOT_DIR / "app").is_dir():
        break
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# ── 关键：切换到项目根目录，确保 pydantic-settings 能找到 .env ──
os.chdir(str(ROOT_DIR))

# ── 导入应用配置（读取 .env 或 app/core/config.py 默认值）──
from app.core.config import settings

from evaluation.retrieval_metrics.metrics import (
    context_redundancy_at_k,
    mrr,
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
)
from evaluation.retrieval_metrics.matching import (
    RetrievedItem,
    RelevantSource,
    build_retrieved_item,
    match_retrieved_to_relevant_sources,
    relevant_source_from_dict,
)
from evaluation.retrieval_metrics.evaluator import evaluate_retrieval_case

sns.set_theme(style="whitegrid", context="notebook", palette="muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "font.sans-serif": ["Arial", "DejaVu Sans", "Helvetica"],
})
%matplotlib inline

# ── 打印当前环境配置，方便确认 ──
print("=" * 58)
print("📋 当前环境配置（来自 .env 或 app/core/config.py）")
print(f"   项目根目录: {ROOT_DIR}")
print(f"   .env 存在: {(ROOT_DIR / '.env').exists()}")
print("-" * 58)
print(f"  embedding_model:    {settings.embedding_model}")
print(f"  embedding_base_url: {settings.embedding_base_url}")
print(f"  chunk_size:         {settings.chunk_size}")
print(f"  chunk_overlap:      {settings.chunk_overlap}")
print(f"  qdrant_collection:  {settings.qdrant_collection}")
print(f"  qdrant_url:         {settings.qdrant_url}")
print(f"  reranker_type:      {settings.reranker_type}")
print(f"  reranker_model:     {settings.reranker_model}")
print(f"  reranker_device:    {settings.reranker_device}")
print("=" * 58)
print()

## 2. 运行配置

In [ ]:
DATASET_PATH = "evaluation/datasets/golden_retrieval.example.jsonl"
OUTPUT_DIR = Path("evaluation/results")
TOP_K = 5
EXPERIMENT_NAME = "rerank"
USE_QUERY_PROCESSOR = True
LIVE_MODE = True

# ── Rerank 配置 ──
RERANK_ENABLED = True          # 改为 True 以启用 rerank
RERANKER_TYPE = "cross_encoder" # "cross_encoder" | "hybrid"
RERANK_TOP_N = 20               # 检索阶段取几条候选给 rerank

dataset_file = Path(DATASET_PATH)
if not dataset_file.exists():
    raise FileNotFoundError(f"找不到 golden dataset: {dataset_file.resolve()}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"数据集: {dataset_file.resolve()}")
print(f"Top-K: {TOP_K}")
print(f"实验名: {EXPERIMENT_NAME}  ← 换策略时改这个名字")
print(f"模式: {'真实检索 (需要 Qdrant + Embedding 服务)' if LIVE_MODE else '模拟检索 (可离线运行)'}")
if RERANK_ENABLED:
    print(f"Rerank: 启用 ({RERANKER_TYPE}), 检索 {RERANK_TOP_N} 条 → 保留 {TOP_K} 条")
else:
    print(f"Rerank: 禁用 (vector-only baseline)")
print()

## 3. 加载 Golden Dataset

In [ ]:
def load_dataset(path: str) -> list[dict[str, Any]]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            if "question" not in rec:
                raise ValueError(f"记录缺少 question 字段: {rec.get('id', '?')}")
            records.append(rec)
    return records

dataset = load_dataset(DATASET_PATH)
print(f"已加载 {len(dataset)} 条 golden questions\n")

for rec in dataset[:5]:
    n_sources = len(rec.get("relevant_sources", []))
    print(f"  [{rec['id']}] {rec['question'][:70]}...  ({n_sources} 个相关来源)")
if len(dataset) > 5:
    print(f"  ... 还有 {len(dataset) - 5} 条")

## 4. 运行检索评估

In [ ]:
def run_live_retrieval(question: str, top_k: int) -> list[RetrievedItem]:
    """通过真实 Qdrant vectorstore 和 embedding 服务进行检索，可选 rerank。"""
    from app.rag.retriever import retrieve_relevant_documents
    from app.rag.query_processor import process_query

    # 用 "or question" 作为 fallback，避免 greeting 类查询返回 None
    retrieval_query = (process_query(question)["rewritten_query"] or question) if USE_QUERY_PROCESSOR else question

    # 如果启用 rerank，检索更多候选
    retrieval_k = RERANK_TOP_N if RERANK_ENABLED else top_k
    results = retrieve_relevant_documents(retrieval_query, top_k=retrieval_k)

    # Rerank
    if RERANK_ENABLED and len(results) > top_k:
        from app.rag.reranker import CrossEncoderReranker, HybridFusionReranker

        if RERANKER_TYPE == "cross_encoder":
            reranker = CrossEncoderReranker()
        elif RERANKER_TYPE == "hybrid":
            reranker = HybridFusionReranker()
        else:
            reranker = None

        if reranker is not None:
            documents_for_rerank = [doc for doc, _score in results]
            results = reranker.rerank(
                query=retrieval_query,
                documents=documents_for_rerank,
                top_k=top_k,
            )

    return [
        build_retrieved_item(content=doc.page_content, metadata=doc.metadata, score=float(score) if score else None)
        for doc, score in results
    ]


def run_demo_retrieval(record: dict[str, Any], top_k: int) -> list[RetrievedItem]:
    """模拟检索结果，生成不同质量等级的检索效果，用于离线测试。"""
    rng = np.random.RandomState(hash(record["id"]) % (2**31))
    sources = record.get("relevant_sources", [])

    items: list[RetrievedItem] = []

    for i, src in enumerate(sources):
        file_path = src.get("file_path", "unknown.md")
        text_hint = src.get("text", "relevant content")
        relevance = src.get("relevance", 1.0)

        if rng.random() < 0.55 + relevance * 0.12:
            rank_pos = rng.randint(0, min(top_k, len(sources) + 2))
            content = f"[模拟 chunk - {file_path}] 本文档详细讨论了 {text_hint}，提供了全面的主题覆盖及示例和最佳实践。"
            items.append(build_retrieved_item(
                content=content,
                metadata={"file_path": file_path, "chunk_index": i},
                score=1.0 - rank_pos * 0.12,
            ))

    distractor_topics = [
        "办公室零食政策", "2024 节假日日历", "食堂菜单",
        "v3.2.1 发布说明", "停车证续期",
        "团队 offsite 计划", "打印机故障排除指南",
        "差旅报销表", "电梯维护时间表",
    ]
    while len(items) < top_k:
        topic = distractor_topics[len(items) % len(distractor_topics)]
        items.append(build_retrieved_item(
            content=f"[模拟干扰项] 本页面涵盖 {topic} 及相关行政流程。",
            metadata={"file_path": f"noise/{topic}.md", "chunk_index": 99},
            score=0.3 - len(items) * 0.03,
        ))

    items.sort(key=lambda x: x.score if x.score is not None else 0.0, reverse=True)
    return items[:top_k]


# ── 检索时多拿一些（供 §11 Top-K 敏感度分析），评估仍按 TOP_K ──
RETRIEVAL_K = max(TOP_K, max(RERANK_TOP_N if RERANK_ENABLED else 0, 10))

print(f"正在以 {'真实' if LIVE_MODE else '模拟'} 模式运行评估...")
if RERANK_ENABLED:
    print(f"  Rerank: {RERANKER_TYPE}, 检索 {RERANK_TOP_N} 条 → 保留 {TOP_K} 条")
print(f"  检索 K = {RETRIEVAL_K}（取更多结果供 §11 敏感度分析），评估 K = {TOP_K}\n")

per_question_results = []
for rec in dataset:
    question = rec["question"]
    relevant_sources = [relevant_source_from_dict(s) for s in rec.get("relevant_sources", [])]

    if LIVE_MODE:
        retrieved_items = run_live_retrieval(question, RETRIEVAL_K)
    else:
        retrieved_items = run_demo_retrieval(rec, RETRIEVAL_K)

    # 评估始终用 TOP_K，不影响主指标
    eval_result = evaluate_retrieval_case(retrieved_items, relevant_sources, TOP_K)

    per_question_results.append({
        "id": rec["id"],
        "question": question,
        "num_relevant": len(relevant_sources),
        "num_retrieved": len(retrieved_items),
        "evaluation": eval_result.to_dict(),
    })
    print(f"  [{rec['id']}] recall@{TOP_K}={eval_result.core_metrics[f'recall@{TOP_K}']:.2f}  "
          f"precision@{TOP_K}={eval_result.core_metrics[f'precision@{TOP_K}']:.2f}  "
          f"mrr={eval_result.core_metrics['mrr']:.2f}")

print(f"\n共评估 {len(per_question_results)} 个问题。")

## 5. 构建结果 DataFrame

In [ ]:
rows = []
for r in per_question_results:
    ev = r["evaluation"]
    rows.append({
        "id": r["id"],
        "问题": r["question"][:60],
        "相关来源数": r["num_relevant"],
        f"recall@{TOP_K}": ev["core_metrics"][f"recall@{TOP_K}"],
        f"precision@{TOP_K}": ev["core_metrics"][f"precision@{TOP_K}"],
        "mrr": ev["core_metrics"]["mrr"],
        f"ndcg@{TOP_K}": ev["core_metrics"][f"ndcg@{TOP_K}"],
        f"上下文冗余@{TOP_K}": ev["context_quality"][f"context_redundancy@{TOP_K}"],
        f"无关率@{TOP_K}": ev["context_quality"][f"irrelevant_rate@{TOP_K}"],
        f"重复率@{TOP_K}": ev["context_quality"][f"duplicate_rate@{TOP_K}"],
    })

df = pd.DataFrame(rows)
df.set_index("id", inplace=True)
df

## 6. 聚合指标统计

In [ ]:
def aggregate_metrics(results: list[dict[str, Any]]) -> dict[str, dict[str, float]]:
    """对所有问题的 core_metrics 和 context_quality 计算均值与标准差。"""
    agg: dict[str, dict[str, float]] = {}
    for group in ("core_metrics", "context_quality"):
        metric_names = results[0]["evaluation"][group].keys()
        agg[group] = {}
        for name in metric_names:
            vals = [r["evaluation"][group][name] for r in results]
            agg[group][f"{name}_mean"] = mean(vals)
            agg[group][f"{name}_std"] = stdev(vals) if len(vals) > 1 else 0.0
    return agg

agg = aggregate_metrics(per_question_results)

print("=" * 56)
print(f"{'检索评估汇总':^50}")
print(f"{'数据集: ' + DATASET_PATH:^56}")
print(f"{'评估问题数: ' + str(len(per_question_results)):^56}")
print(f"{'Top-K: ' + str(TOP_K):^56}")
print("=" * 56)
print()
print("--- 核心检索指标 ---")
for k, v in agg["core_metrics"].items():
    print(f"  {k:<28s} {v:>8.4f}")
print()
print("--- 上下文质量指标 ---")
for k, v in agg["context_quality"].items():
    print(f"  {k:<28s} {v:>8.4f}")

## 7. 逐题核心指标

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

# ===== 中文字体设置：Mac 推荐 =====
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang HK", "Heiti TC"]
plt.rcParams["axes.unicode_minus"] = False

# ===== 基础设置 =====
core_cols = [f"recall@{TOP_K}", f"precision@{TOP_K}", "mrr", f"ndcg@{TOP_K}"]
core_labels = [f"Recall@{TOP_K}", f"Precision@{TOP_K}", "MRR", f"NDCG@{TOP_K}"]

plot_df = df.copy()
plot_df.index = plot_df.index.astype(str)

n_questions = len(plot_df)

# 高度根据问题数量自动调整
fig_height = max(8, n_questions * 0.45)

fig, axes = plt.subplots(
    2, 2,
    figsize=(18, fig_height),
    sharey=True
)

fig.suptitle(
    f"Per-question Core Retrieval Metrics (Top-{TOP_K})",
    fontsize=18,
    fontweight="bold",
    y=1.02
)

# 颜色：统一一点，不要每一题一个颜色，阅读更稳
bar_color = "#4C78A8"
mean_color = "#E45756"

for ax, col, label in zip(axes.flat, core_cols, core_labels):
    values = plot_df[col].fillna(0)

    bars = ax.barh(
        plot_df.index,
        values,
        color=bar_color,
        edgecolor="white",
        linewidth=0.6,
        height=0.65
    )

    # 均值线
    mean_val = values.mean()
    ax.axvline(
        mean_val,
        color=mean_color,
        linestyle="--",
        linewidth=1.8,
        label=f"Mean: {mean_val:.3f}"
    )

    # 每个 bar 右侧显示数值
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(
                val + 0.015,
                bar.get_y() + bar.get_height() / 2,
                f"{val:.2f}",
                va="center",
                fontsize=8
            )

    ax.set_title(label, fontsize=14, fontweight="bold", pad=10)
    ax.set_xlim(0, 1.08)

    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

    # 网格线更淡一点
    ax.grid(axis="x", linestyle="--", alpha=0.35)
    ax.set_axisbelow(True)

    # 去掉多余边框
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(loc="lower right", fontsize=9, frameon=True)

# y 轴反转：让 q01 在上面，阅读更自然
for ax in axes.flat:
    ax.invert_yaxis()
    ax.tick_params(axis="y", labelsize=9)
    ax.tick_params(axis="x", labelsize=9)

plt.tight_layout()
plt.show()

## 8. 逐题上下文质量

In [ ]:
context_cols = [f"上下文冗余@{TOP_K}", f"无关率@{TOP_K}", f"重复率@{TOP_K}"]
context_labels = [f"上下文冗余@{TOP_K}", f"无关率@{TOP_K}", f"重复率@{TOP_K}"]

bar_color_context = "#E57373"
mean_color = "#C62828"

fig, axes = plt.subplots(1, 3, figsize=(18, max(5, n_questions * 0.35)))
fig.suptitle(f"逐题上下文质量 (top-{TOP_K})", fontsize=16, fontweight="bold", y=1.02)

for ax, col, label in zip(axes, context_cols, context_labels):
    values = df[col].fillna(0)
    bars = ax.barh(df.index, values, color=bar_color_context, edgecolor="white", linewidth=0.5, height=0.7)
    mean_val = values.mean()
    ax.axvline(mean_val, color=mean_color, linestyle="--", linewidth=1.5, label=f"均值: {mean_val:.3f}")
    ax.set_title(label, fontweight="bold")
    ax.set_xlim(0, 1.05)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.legend(loc="lower right", fontsize=8)
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 9. 聚合指标总览

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("检索质量聚合总览", fontsize=16, fontweight="bold")

core_means = {k.replace("_mean", ""): v for k, v in agg["core_metrics"].items() if k.endswith("_mean")}
core_stds = {k.replace("_mean", ""): v for k, v in agg["core_metrics"].items() if k.endswith("_std")}

names = list(core_means.keys())
means = list(core_means.values())
stds = list(core_stds.values())
palette = sns.color_palette("Blues_d", len(names))

bars = axes[0].bar(names, means, yerr=stds, color=palette, edgecolor="white", linewidth=1.2,
                   capsize=6, error_kw={"linewidth": 1.2})
axes[0].set_title(f"核心检索指标 (top-{TOP_K})", fontweight="bold")
axes[0].set_ylim(0, 1.1)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
for bar, val in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
                 f"{val:.3f}", ha="center", fontsize=10, fontweight="bold")

cq_means = {k.replace("_mean", ""): v for k, v in agg["context_quality"].items() if k.endswith("_mean")}
cq_stds = {k.replace("_mean", ""): v for k, v in agg["context_quality"].items() if k.endswith("_std")}

names_cq = list(cq_means.keys())
means_cq = list(cq_means.values())
stds_cq = list(cq_stds.values())
palette_cq = sns.color_palette("Oranges_d", len(names_cq))

bars_cq = axes[1].bar(names_cq, means_cq, yerr=stds_cq, color=palette_cq, edgecolor="white",
                      linewidth=1.2, capsize=6, error_kw={"linewidth": 1.2})
axes[1].set_title(f"上下文质量 (top-{TOP_K})", fontweight="bold")
axes[1].set_ylim(0, 1.1)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
for bar, val in zip(bars_cq, means_cq):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
                 f"{val:.3f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## 10. 雷达图 — 检索质量画像

In [ ]:
def plot_radar(ax, labels: list[str], values: list[float], title: str, color: str,
               linewidth: int = 2, alpha: float = 0.25) -> None:
    """在指定坐标轴绘制单张雷达图。"""
    n = len(labels)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]
    values += values[:1]

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["20%", "40%", "60%", "80%", "100%"], fontsize=7, color="grey")
    ax.fill(angles, values, color=color, alpha=alpha)
    ax.plot(angles, values, color=color, linewidth=linewidth)
    for a, v in zip(angles[:-1], values[:-1]):
        ax.scatter(a, v, color=color, s=40, zorder=5)
    ax.set_title(title, fontweight="bold", pad=20)


radar_labels = [
    f"Recall@{TOP_K}",
    f"Precision@{TOP_K}",
    "MRR",
    f"NDCG@{TOP_K}",
    f"1 − 无关率@{TOP_K}",
    f"1 − 重复率@{TOP_K}",
]
radar_values = [
    df[f"recall@{TOP_K}"].mean(),
    df[f"precision@{TOP_K}"].mean(),
    df["mrr"].mean(),
    df[f"ndcg@{TOP_K}"].mean(),
    1.0 - df[f"无关率@{TOP_K}"].mean(),
    1.0 - df[f"重复率@{TOP_K}"].mean(),
]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"projection": "polar"})
plot_radar(ax, radar_labels, radar_values, "检索质量画像", "#2c3e50")
plt.tight_layout()
plt.show()

## 11. Top-K 敏感度分析

In [ ]:
def compute_metrics_at_k(k: int) -> dict[str, float]:
    """用已有的检索结果重新计算不同 K 值下的聚合指标。"""
    vals: dict[str, list[float]] = {
        f"recall@{k}": [], f"precision@{k}": [], f"ndcg@{k}": [],
        f"context_redundancy@{k}": [], f"irrelevant_rate@{k}": [], f"duplicate_rate@{k}": [],
    }
    for r in per_question_results:
        ev = r["evaluation"]
        retrieved_ids = ev["retrieved_ids"]
        relevant_ids = set(ev["matched_relevant_ids"])
        relevance_scores_flat = {rid: 1.0 for rid in relevant_ids}

        vals[f"recall@{k}"].append(recall_at_k(retrieved_ids, relevant_ids, k))
        vals[f"precision@{k}"].append(precision_at_k(retrieved_ids, relevant_ids, k))
        vals[f"ndcg@{k}"].append(ndcg_at_k(retrieved_ids, k=k, relevance_scores=relevance_scores_flat))
        cq = context_redundancy_at_k(retrieved_ids, relevant_ids, k)
        vals[f"context_redundancy@{k}"].append(cq[f"context_redundancy@{k}"])
        vals[f"irrelevant_rate@{k}"].append(cq[f"irrelevant_rate@{k}"])
        vals[f"duplicate_rate@{k}"].append(cq[f"duplicate_rate@{k}"])

    return {name: mean(vlist) for name, vlist in vals.items()}


# ── 从实际检索结果推断可分析的 K 范围 ──
max_retrieved = len(per_question_results[0]["evaluation"]["retrieved_ids"])
k_values = list(range(1, max_retrieved + 1))

sensitivity: dict[str, list[float]] = {
    k: [] for k in ["recall", "precision", "ndcg", "redundancy", "irrelevant", "duplicate"]
}

for k in k_values:
    m = compute_metrics_at_k(k)
    for key in sensitivity:
        sensitivity[key].append(m.get(f"{key}@{k}" if key == "redundancy" else (
            f"context_{key}@{k}" if key in ("irrelevant", "duplicate") else f"{key}@{k}"), 0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"不同 Top-K 下的指标变化（基于 {max_retrieved} 条实际检索结果）", fontsize=16, fontweight="bold")

ax = axes[0]
ax.plot(k_values, sensitivity["recall"], "o-", label="Recall@K", linewidth=2, markersize=6)
ax.plot(k_values, sensitivity["precision"], "s-", label="Precision@K", linewidth=2, markersize=6)
ax.plot(k_values, sensitivity["ndcg"], "^-", label="NDCG@K", linewidth=2, markersize=6)
ax.set_xlabel("K")
ax.set_ylabel("得分")
ax.set_title("核心检索指标", fontweight="bold")
ax.legend()
ax.set_xticks(k_values)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

ax = axes[1]
ax.plot(k_values, sensitivity["redundancy"], "o-", label="上下文冗余@K", linewidth=2, markersize=6)
ax.plot(k_values, sensitivity["irrelevant"], "s-", label="无关率@K", linewidth=2, markersize=6)
ax.plot(k_values, sensitivity["duplicate"], "^-", label="重复率@K", linewidth=2, markersize=6)
ax.set_xlabel("K")
ax.set_ylabel("比率")
ax.set_title("上下文质量", fontweight="bold")
ax.legend()
ax.set_xticks(k_values)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

plt.tight_layout()
plt.show()

## 12. Recall 分布与精度关系

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bins_recall = np.linspace(0, 1, 11)
ax.hist(df[f"recall@{TOP_K}"], bins=bins_recall, color="#3498db", edgecolor="white", linewidth=1.2, alpha=0.85)
ax.axvline(df[f"recall@{TOP_K}"].mean(), color="#e74c3c", linestyle="--", linewidth=2,
           label=f"均值: {df[f'recall@{TOP_K}'].mean():.3f}")
ax.set_title(f"Recall@{TOP_K} 分布", fontweight="bold")
ax.set_xlabel(f"Recall@{TOP_K}")
ax.set_ylabel("问题数量")
ax.legend()
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

ax = axes[1]
scatter = ax.scatter(
    df[f"recall@{TOP_K}"],
    df[f"precision@{TOP_K}"],
    s=df[f"ndcg@{TOP_K}"] * 200 + 20,
    c=df[f"上下文冗余@{TOP_K}"],
    cmap="RdYlGn_r",
    edgecolors="#2c3e50",
    linewidth=0.5,
    alpha=0.85,
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label(f"上下文冗余@{TOP_K}", fontsize=9)
ax.set_xlabel(f"Recall@{TOP_K}")
ax.set_ylabel(f"Precision@{TOP_K}")
ax.set_title(f"Recall vs. Precision (气泡大小 ∝ NDCG@{TOP_K})", fontweight="bold")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.axhline(0.5, color="grey", linestyle=":", alpha=0.4)
ax.axvline(0.5, color="grey", linestyle=":", alpha=0.4)

ax.text(0.25, 0.25, "低召回 低精度", ha="center", fontsize=7, color="grey", alpha=0.7)
ax.text(0.75, 0.25, "高召回 低精度", ha="center", fontsize=7, color="grey", alpha=0.7)
ax.text(0.75, 0.75, "高召回 高精度", ha="center", fontsize=7, color="grey", alpha=0.7)
ax.text(0.25, 0.75, "低召回 高精度", ha="center", fontsize=7, color="grey", alpha=0.7)

plt.tight_layout()
plt.show()

## 13. 指标相关性热力图

In [ ]:
metric_cols = [
    f"recall@{TOP_K}", f"precision@{TOP_K}", "mrr", f"ndcg@{TOP_K}",
    f"上下文冗余@{TOP_K}", f"无关率@{TOP_K}", f"重复率@{TOP_K}",
]
corr = df[metric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
ax.set_title(f"指标相关性矩阵 (top-{TOP_K})", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

## 14. 导出报告

In [ ]:
from datetime import datetime
import json
import re

# ── 从当前环境读取配置，写入报告快照 ──
from app.core.config import settings

# 当前时间
now = datetime.now()

date_str = now.strftime("%Y-%m-%d")
time_str = now.strftime("%I-%M%p")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pattern = re.compile(rf"^{re.escape(date_str)}_.*_(\d+)\.json$")
existing_runs = []

for file in OUTPUT_DIR.glob(f"{date_str}_*.json"):
    match = pattern.match(file.name)
    if match:
        existing_runs.append(int(match.group(1)))

run_number = max(existing_runs, default=0) + 1
output_filename = f"{date_str}_{time_str}_{run_number}.json"
output_path = OUTPUT_DIR / output_filename

report = {
    "report_schema": "retrieval-eval-v1",
    "experiment_name": EXPERIMENT_NAME,
    "created_at": now.isoformat(timespec="seconds"),
    "evaluation_layer": "retrieval",
    "dataset": DATASET_PATH,
    "top_k": TOP_K,
    "mode": "live" if LIVE_MODE else "demo",
    "num_questions": len(per_question_results),
    # ══════════════════════════════════════════════════════
    # 关键：把当前 .env / config 快照写入报告，
    # 这样后续对比时能自动显示 chunk_size / embedding_model
    # ══════════════════════════════════════════════════════
    "settings_snapshot": {
        "qdrant_collection": settings.qdrant_collection,
        "chunk_size": settings.chunk_size,
        "chunk_overlap": settings.chunk_overlap,
        "embedding_model": settings.embedding_model,
    },
    "aggregate": {
        "core_metrics": {k: round(v, 4) for k, v in agg["core_metrics"].items()},
        "context_quality": {k: round(v, 4) for k, v in agg["context_quality"].items()},
    },
    "per_question": [
        {
            "id": r["id"],
            "question": r["question"],
            "core_metrics": r["evaluation"]["core_metrics"],
            "context_quality": r["evaluation"]["context_quality"],
        }
        for r in per_question_results
    ],
}

# ── Rerank 配置（如果本次运行启用了）──
if RERANK_ENABLED:
    report["rerank_config"] = {
        "reranker_type": RERANKER_TYPE,
        "reranker_model": settings.reranker_model,
        "rerank_top_n": RERANK_TOP_N,
        "final_top_k": TOP_K,
        "max_chars": settings.reranker_max_chars,
        "device": settings.reranker_device,
    }

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
    f.write("\n")

print(f"报告已导出: {output_path.resolve()}")
print(f"今日第几次运行: {run_number}")
print(f"配置快照: chunk={settings.chunk_size}/{settings.chunk_overlap}, emb={settings.embedding_model}")
if RERANK_ENABLED:
    print(f"Rerank: {RERANKER_TYPE}, model={settings.reranker_model}, {RERANK_TOP_N}→{TOP_K}")
print(f"文件大小: {output_path.stat().st_size:,} bytes")

## 15. 自动诊断

In [ ]:
recall_mean = df[f"recall@{TOP_K}"].mean()
precision_mean = df[f"precision@{TOP_K}"].mean()
mrr_mean = df["mrr"].mean()
redundancy_mean = df[f"上下文冗余@{TOP_K}"].mean()
ndcg_mean = df[f"ndcg@{TOP_K}"].mean()

diagnoses = []

if recall_mean > 0.7 and precision_mean < 0.4:
    diagnoses.append("高召回低精度 → 系统能找到相关文档，但上下文窗口噪声较多。建议增大 top-K 或引入 reranker。")
elif precision_mean > 0.7 and recall_mean < 0.4:
    diagnoses.append("高精度低召回 → 上下文干净但漏召回风险大。建议调整 chunk_size 或引入混合检索。")

if mrr_mean < 0.5:
    diagnoses.append("MRR 偏低 → 第一个相关文档排得太靠后，LLM 可能忽略关键证据。建议优化 embedding 模型或加入 reranker。")

if redundancy_mean > 0.4:
    diagnoses.append("上下文冗余偏高 → top-K 中无关或重复内容过多，浪费上下文窗口。检查 chunk_overlap 和去重策略。")

if ndcg_mean > 0.7:
    diagnoses.append("NDCG 表现优秀 → 高价值证据排名靠前，排序质量良好。")

if not diagnoses:
    diagnoses.append("指标表现均衡。随着知识库增长，继续监控即可。")

print("=" * 60)
print("诊断结果")
print("=" * 60)
for i, d in enumerate(diagnoses, 1):
    print(f"\n  {i}. {d}")
print()

---
## 16. 策略对比（Embedding / Chunking / Top-K / Rerank 通用）

以下单元格用于对比不同配置下的评估结果，适用于任何你想对比的策略维度。

### 对比测试流程（以 Rerank 为例）

```bash
# ============================================================
# 🔴 第 1 步：配置 .env 中的 rerank 参数
# ============================================================
# RERANKER_TYPE=cross_encoder
# RERANKER_MODEL=BAAI/bge-reranker-base
# RERANKER_CANDIDATE_TOP_N=20
# RERANKER_FINAL_TOP_K=5

# ============================================================
# 第 2 步：运行评估 CLI，生成 baseline（无 rerank）
# ============================================================
python evaluation/run_retrieval_eval.py \
  --dataset evaluation/datasets/golden_retrieval.example.jsonl \
  --top-k 5 \
  --experiment-name baseline-vector-only

# ============================================================
# 第 3 步：运行评估 CLI，生成 rerank 实验报告
# ============================================================
python evaluation/run_retrieval_eval.py \
  --dataset evaluation/datasets/golden_retrieval.example.jsonl \
  --top-k 5 \
  --use-reranker --reranker-type cross_encoder --rerank-top-n 20 \
  --experiment-name rerank-bge-base-top20

# ============================================================
# 第 4 步：回到本 notebook，运行以下单元格
# ============================================================
# 自动加载所有报告并生成对比图表
```

**对比其他维度同理：**
- 对比 embedding 模型：改 `.env` → 清空 Qdrant + checksum DB → 重新摄入 → 评估
- 对比 chunk_size：改 `.env` → 清空 Qdrant → 重新摄入 → 评估
- 对比 top-k：改 `--top-k` 参数即可（无需重新摄入）
- 对比 query_processor：加 `--use-query-processor` 标志
- **对比 rerank 策略**：用 `--use-reranker` + `--reranker-type` 切换 cross_encoder / hybrid
- **对比 rerank_top_n**：用 `--rerank-top-n 10/20/30` 测试不同候选数量

**Rerank 实验推荐矩阵：**
```bash
# Experiment 1: baseline (no rerank)
--experiment-name baseline-vector-only

# Experiment 2: Cross-Encoder with top-20
--use-reranker --reranker-type cross_encoder --rerank-top-n 20 \
--experiment-name rerank-cross-encoder-top20

# Experiment 3: Cross-Encoder with top-30
--use-reranker --reranker-type cross_encoder --rerank-top-n 30 \
--experiment-name rerank-cross-encoder-top30

# Experiment 4: Hybrid fusion (lightweight fallback)
--use-reranker --reranker-type hybrid --rerank-top-n 20 \
--experiment-name rerank-hybrid-top20
```

**关键提醒（必读）：**
- Rerank 实验**不需要**重新摄入文档，直接运行评估即可
- `scripts/clear_qdrant.py` 只删 Qdrant 集合，**不删 `data/ingestion_state.db`**
- 如果不清 checksum 数据库，`ingest.py` 会认为所有文件"已摄入、未变更"，全部跳过
- 所以换 embedding 模型后必须**同时清 Qdrant + 删 checksum DB**：`python scripts/clear_qdrant.py && rm -f data/ingestion_state.db`
- 如果只改 chunk_size 而 embedding 模型不变，理论上不需要清 Qdrant，但也建议清，保证每轮测试的向量完全对应

In [ ]:
from pathlib import Path
from typing import Any
import json
import re


def parse_report_filename(file_path: Path) -> dict[str, str]:
    """解析 report 文件名。

    目标格式：
    2026-05-18_04-16PM_2.json

    解析结果：
    date = 2026-05-18
    time = 04-16PM
    run_number = 2
    """
    report_id = file_path.stem

    pattern = re.compile(
        r"^(?P<date>\d{4}-\d{2}-\d{2})_"
        r"(?P<time>\d{2}-\d{2}[AP]M)_"
        r"(?P<run_number>\d+)$"
    )

    match = pattern.match(report_id)

    if not match:
        return {
            "report_id": report_id,
            "date": "unknown",
            "time": "unknown",
            "run_number": "unknown",
        }

    return {
        "report_id": report_id,
        "date": match.group("date"),
        "time": match.group("time"),
        "run_number": match.group("run_number"),
    }


def load_all_reports(results_dir: str = "evaluation/results") -> dict[str, dict[str, Any]]:
    """加载所有 retrieval-eval-v1 JSON 报告。"""
    reports_dir = Path(results_dir)

    if not reports_dir.exists():
        print(f"⚠️ results 目录不存在: {reports_dir.resolve()}")
        return {}

    reports: dict[str, dict[str, Any]] = {}

    json_files = sorted(reports_dir.glob("*.json"))

    for file_path in json_files:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            if data.get("report_schema") != "retrieval-eval-v1":
                continue

            filename_info = parse_report_filename(file_path)
            report_id = filename_info["report_id"]

            experiment_name = data.get("experiment_name", "unknown")

            # ══════════════════════════════════════════════════════
            # 从报告内部提取配置信息，用于展示
            # ══════════════════════════════════════════════════════
            snapshot = data.get("settings_snapshot", {})
            chunk_size = snapshot.get("chunk_size", "?")
            chunk_overlap = snapshot.get("chunk_overlap", "?")
            embedding_model = snapshot.get("embedding_model", "?")

            # ── Rerank 配置 ──
            rerank_config = data.get("rerank_config")
            rerank_label = "none"
            if rerank_config:
                rerank_label = (
                    f"{rerank_config.get('reranker_type', '?')}"
                    f" | {rerank_config.get('reranker_model', '?')[:24]}"
                    f" | top{rerank_config.get('rerank_top_n', '?')}→{rerank_config.get('final_top_k', '?')}"
                )

            data["_report_id"] = report_id
            data["_file_name"] = file_path.name
            data["_file_path"] = str(file_path)
            data["_date"] = filename_info["date"]
            data["_time"] = filename_info["time"]
            data["_run_number"] = filename_info["run_number"]
            data["_chunk_size"] = chunk_size
            data["_chunk_overlap"] = chunk_overlap
            data["_embedding_model"] = embedding_model
            data["_rerank_config"] = rerank_config
            data["_rerank_label"] = rerank_label

            # ── 紧凑标签（饼图 / 柱状图 legend 用）──
            label_parts = [experiment_name, f"chunk={chunk_size}", embedding_model[:28]]
            if rerank_config:
                label_parts.append(f"rerank={rerank_config.get('reranker_type','?')}")
            data["_short_label"] = " | ".join(label_parts)

            # ── 完整标签（对比表用）──
            display = (
                f"[{filename_info['date']} {filename_info['time']}] "
                f"Run {filename_info['run_number']} — "
                f"chunk={chunk_size}/{chunk_overlap}, "
                f"emb={embedding_model}"
            )
            if rerank_config:
                display += f", rerank={rerank_label}"
            data["_display_name"] = display

            reports[report_id] = data

        except json.JSONDecodeError:
            print(f"跳过 {file_path.name}: JSON 格式错误")
            continue
        except OSError as e:
            print(f"跳过 {file_path.name}: 文件读取失败，{e}")
            continue

    return reports


reports = load_all_reports()

if not reports:
    print("⚠️ 未在 evaluation/results/ 目录中找到可用实验报告。")
    print()
    print("请先运行 retrieval evaluation，生成 JSON report。")
    print()
    print("当前期望路径是：")
    print(f"  {Path('evaluation/results').resolve()}")

else:
    print(f"已加载 {len(reports)} 份实验报告:")
    print()

    for report_id, report in reports.items():
        rerank_info = f"  rerank={report['_rerank_label']}" if report.get("_rerank_config") else ""
        print(f"  📄 {report['_display_name']}")
        print(f"     指标: "
              f"recall@5={report['aggregate']['core_metrics'].get('recall@5_mean', 'N/A'):.3f}, "
              f"precision@5={report['aggregate']['core_metrics'].get('precision@5_mean', 'N/A'):.3f}, "
              f"mrr={report['aggregate']['core_metrics'].get('mrr_mean', 'N/A'):.3f}, "
              f"ndcg@5={report['aggregate']['core_metrics'].get('ndcg@5_mean', 'N/A'):.3f}")
        print()

### 16a. 策略配置与指标汇总

每一行代表一次实验，自动记录当时使用的 `chunk_size / chunk_overlap / embedding_model`。

In [ ]:
def extract_aggregate_means(report: dict[str, Any]) -> dict[str, float]:
    """从报告中提取聚合均值 (去掉 _mean 后缀)。"""
    agg = report["aggregate"]
    core = {k.replace("_mean", ""): v for k, v in agg["core_metrics"].items() if k.endswith("_mean")}
    cq = {k.replace("_mean", ""): v for k, v in agg["context_quality"].items() if k.endswith("_mean")}
    return {**core, **cq}


comparison_rows = []
for name, report in reports.items():
    metrics = extract_aggregate_means(report)

    # ── 自动从报告里提取配置信息 ──
    metrics["Chunk"] = f"{report.get('_chunk_size', '?')}/{report.get('_chunk_overlap', '?')}"
    metrics["Embedding Model"] = report.get("_embedding_model", "?")
    metrics["Rerank"] = report.get("_rerank_label", "none")

    # 保持 _short_label 给后续图表用
    metrics["实验名称"] = name
    metrics["_short_label"] = report.get("_short_label", name)

    comparison_rows.append(metrics)

comparison_df = pd.DataFrame(comparison_rows)

# ── 列顺序：配置信息在前，指标在后 ──
meta_cols = ["实验名称", "Chunk", "Embedding Model", "Rerank"]
metric_cols = [c for c in comparison_df.columns if c not in meta_cols and not c.startswith("_")]
comparison_df = comparison_df[meta_cols + metric_cols]
comparison_df.set_index("实验名称", inplace=True)

# 按 NDCG 降序排列
ndcg_col = [c for c in comparison_df.columns if c.startswith("ndcg@")][0]
comparison_df.sort_values(ndcg_col, ascending=False, inplace=True)

# ── 颜色编码：核心指标越高越绿，冗余指标越低越绿 ──
core_cols = [c for c in metric_cols if not c.startswith("context_") and not c.startswith("irrelevant_") and not c.startswith("duplicate_")]
cq_cols = [c for c in metric_cols if c.startswith("context_") or c.startswith("irrelevant_") or c.startswith("duplicate_")]

styled = comparison_df.style \
    .format({**{c: "{:.4f}" for c in core_cols + cq_cols}}) \
    .background_gradient(cmap="RdYlGn", vmin=0, vmax=1, subset=core_cols) \
    .background_gradient(cmap="RdYlGn_r", vmin=0, vmax=1, subset=cq_cols) \
    .set_caption("策略检索指标对比（绿色 = 优，红色 = 差）")

# ── 打印简要解读 ──
best_recall = comparison_df[core_cols[0]].idxmax()
best_precision = comparison_df[[c for c in core_cols if c.startswith("precision@")][0]].idxmax()
best_mrr = comparison_df["mrr"].idxmax()
best_ndcg = comparison_df[ndcg_col].idxmax()

print("🏆 各指标最优策略：")
print(f"   Recall@{TOP_K}:  {best_recall}")
print(f"   Precision@{TOP_K}: {best_precision}")
print(f"   MRR:            {best_mrr}")
print(f"   NDCG@{TOP_K}:    {best_ndcg}")

# ── Rerank 对比提示 ──
rerank_reports = [r for r in reports.values() if r.get("_rerank_config")]
non_rerank_reports = [r for r in reports.values() if not r.get("_rerank_config")]
if rerank_reports and non_rerank_reports:
    print()
    print("📊 Rerank 对比分析：")
    rerank_ndcg = mean([
        r["aggregate"]["core_metrics"].get("ndcg@5_mean", 0)
        for r in rerank_reports
    ])
    baseline_ndcg = mean([
        r["aggregate"]["core_metrics"].get("ndcg@5_mean", 0)
        for r in non_rerank_reports
    ])
    delta = rerank_ndcg - baseline_ndcg
    direction = "↑ improved" if delta > 0 else "↓ decreased"
    print(f"   Rerank NDCG@5:    {rerank_ndcg:.4f}")
    print(f"   Baseline NDCG@5:  {baseline_ndcg:.4f}")
    print(f"   Delta:            {delta:+.4f} ({direction})")
print()

styled

### 16b. 核心指标对比 — 柱状图

In [ ]:
import matplotlib.ticker as mticker

# ── 准备数据 ──
core_metric_names = [c for c in comparison_df.columns if c in [
    f"recall@{TOP_K}", f"precision@{TOP_K}", "mrr", f"ndcg@{TOP_K}"
]]
core_labels = [f"Recall@{TOP_K}", f"Precision@{TOP_K}", "MRR", f"NDCG@{TOP_K}"]

n_strategies = len(comparison_df)
n_metrics = len(core_metric_names)
bar_width = 0.8 / n_strategies

# ── 使用报告中自带的 short_label ──
strategy_labels = [
    comparison_df.loc[name, "_short_label"]
    if "_short_label" in comparison_df.columns
    else str(name)
    for name in comparison_df.index
]

# ── 颜色：使用更专业的配色 ──
cmap = plt.cm.viridis if n_strategies > 2 else plt.cm.Set2
colors = sns.color_palette("Set2", n_strategies) if n_strategies <= 8 else sns.color_palette("viridis", n_strategies)

fig, ax = plt.subplots(figsize=(16, max(6, n_strategies * 1.3)))

x = np.arange(n_metrics)
for i, (idx_name, row) in enumerate(comparison_df.iterrows()):
    vals = [row.get(c, 0) for c in core_metric_names]
    offset = (i - n_strategies / 2 + 0.5) * bar_width
    bars = ax.bar(
        x + offset, vals, bar_width,
        label=strategy_labels[i],
        color=colors[i],
        edgecolor="white",
        linewidth=0.8,
    )
    # 数值标签
    for bar, val in zip(bars, vals):
        if val > 0.02:
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
                f"{val:.2f}", ha="center", fontsize=7, fontweight="bold",
                rotation=90,
            )

ax.set_xticks(x)
ax.set_xticklabels(core_labels, fontsize=12, fontweight="bold")
ax.set_ylim(0, 1.15)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_title(
    f"Core Retrieval Metrics by Strategy (Top-{TOP_K})",
    fontweight="bold", fontsize=16, pad=18,
)
ax.set_ylabel("Score", fontsize=12)
ax.legend(
    loc="upper right",
    fontsize=min(10, 140 // len(strategy_labels[0])),
    frameon=True,
    title="Strategy (experiment | chunk | embedding)",
    title_fontsize=9,
)
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### 16c. 上下文质量对比 — 越低越好

`context_redundancy` = 不相关率 + 重复率。该值越低，说明 top-K 窗口中的内容越干净、浪费越少。

In [ ]:
cq_metric_names = [c for c in comparison_df.columns if c in [
    f"context_redundancy@{TOP_K}", f"irrelevant_rate@{TOP_K}", f"duplicate_rate@{TOP_K}"
]]
cq_labels = [
    f"Context\nRedundancy@{TOP_K}",
    f"Irrelevant\nRate@{TOP_K}",
    f"Duplicate\nRate@{TOP_K}",
]

fig, ax = plt.subplots(figsize=(16, max(6, n_strategies * 1.3)))

x = np.arange(len(cq_metric_names))
for i, (idx_name, row) in enumerate(comparison_df.iterrows()):
    vals = [row.get(c, 0) for c in cq_metric_names]
    offset = (i - n_strategies / 2 + 0.5) * bar_width
    bars = ax.bar(
        x + offset, vals, bar_width,
        label=strategy_labels[i],
        color=colors[i],
        edgecolor="white",
        linewidth=0.8,
    )
    for bar, val in zip(bars, vals):
        if val > 0.02:
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
                f"{val:.2f}", ha="center", fontsize=7, fontweight="bold",
                rotation=90,
            )

ax.set_xticks(x)
ax.set_xticklabels(cq_labels, fontsize=11, fontweight="bold")
ax.set_ylim(0, 1.15)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_title(
    f"Context Quality by Strategy (Top-{TOP_K}) — Lower is Better",
    fontweight="bold", fontsize=16, pad=18,
)
ax.set_ylabel("Rate", fontsize=12)
ax.legend(
    loc="upper right",
    fontsize=min(10, 140 // len(strategy_labels[0])),
    frameon=True,
    title="Strategy (experiment | chunk | embedding)",
    title_fontsize=9,
)
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### 16d. 雷达图叠加 — 检索质量画像

将六维指标归一化到同一张雷达图中。面积越大、越 "圆"，说明该策略的综合检索质量越高。上下文质量指标已作反向处理（取 `1 − 无关率`、`1 − 重复率`），使所有维度均为「越高越好」。

In [ ]:
radar_metric_keys = [
    ("recall@", "Recall"),
    ("precision@", "Precision"),
    ("mrr", "MRR"),
    ("ndcg@", "NDCG"),
]

def _find_key(prefix: str, keys: list[str]) -> str:
    for k in keys:
        if k.startswith(prefix):
            return k
    return prefix

cq_invert_keys = [
    ("irrelevant_rate@", "1 − Irrelevant"),
    ("duplicate_rate@", "1 − Duplicate"),
]

radar_labels_cmp = []
for _, label in radar_metric_keys:
    radar_labels_cmp.append(label)
for _, label in cq_invert_keys:
    radar_labels_cmp.append(label)

all_keys = list(comparison_df.columns)
n_radar_strategies = len(comparison_df)

# ── 为雷达图单独生成颜色（确保可区分）──
radar_colors = sns.color_palette("husl", n_radar_strategies) if n_radar_strategies > 2 else sns.color_palette("Set2", n_radar_strategies)

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={"projection": "polar"})

for i, (idx_name, row) in enumerate(comparison_df.iterrows()):
    values = []
    for prefix, _ in radar_metric_keys:
        k = _find_key(prefix, all_keys)
        values.append(row.get(k, 0))
    for prefix, _ in cq_invert_keys:
        k = _find_key(prefix, all_keys)
        values.append(1.0 - row.get(k, 0))

    label = strategy_labels[i] if i < len(strategy_labels) else str(idx_name)
    plot_radar(ax, radar_labels_cmp, values, "", radar_colors[i],
               linewidth=2.5, alpha=0.08)

ax.set_title("Strategy Comparison — Retrieval Quality Radar", fontweight="bold", pad=25, fontsize=16)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color=radar_colors[i], linewidth=2.5,
           label=strategy_labels[i] if i < len(strategy_labels) else str(name))
    for i, name in enumerate(comparison_df.index)
]
ax.legend(
    handles=legend_elements,
    loc="upper right",
    bbox_to_anchor=(1.45, 1.12),
    fontsize=min(9, 180 // len(strategy_labels[0])),
    frameon=True,
    title="Strategy",
    title_fontsize=10,
)

plt.tight_layout()
plt.show()

### 16e. 权衡分析 — Recall vs. Precision

气泡大小 ∝ NDCG@K，颜色深度 ∝ 上下文冗余。理想策略应靠近右上角（高召回 + 高精度）且颜色浅（低冗余）。

In [ ]:
recall_key = _find_key("recall@", all_keys)
precision_key = _find_key("precision@", all_keys)
redundancy_key = _find_key("context_redundancy@", all_keys)
ndcg_key = _find_key("ndcg@", all_keys)

fig, ax = plt.subplots(figsize=(11, 8))

for i, (idx_name, row) in enumerate(comparison_df.iterrows()):
    ax.scatter(
        row[recall_key], row[precision_key],
        s=350,
        color=radar_colors[i],
        edgecolors="#2c3e50",
        linewidth=1.8,
        alpha=0.92,
        zorder=5,
    )
    label = strategy_labels[i] if i < len(strategy_labels) else str(idx_name)
    ax.annotate(
        label,
        (row[recall_key], row[precision_key]),
        textcoords="offset points",
        xytext=(12, 12),
        fontsize=9,
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.4",
            facecolor="white",
            alpha=0.85,
            edgecolor=radar_colors[i],
            linewidth=1.2,
        ),
    )

ax.set_xlabel(f"Recall@{TOP_K} →", fontsize=13, fontweight="bold")
ax.set_ylabel(f"Precision@{TOP_K} →", fontsize=13, fontweight="bold")
ax.set_title("Strategy Trade-off: Recall vs. Precision", fontweight="bold", fontsize=16, pad=15)
ax.set_xlim(-0.05, 1.08)
ax.set_ylim(-0.05, 1.08)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

# ── 象限分隔线 ──
ax.axhline(0.5, color="grey", linestyle=":", alpha=0.3, linewidth=1)
ax.axvline(0.5, color="grey", linestyle=":", alpha=0.3, linewidth=1)

# ── 象限注解 ──
ax.text(0.03, 0.96, "High Precision\nLow Recall\n" + "─" * 20 + "\nClean but misses docs",
        transform=ax.transAxes, fontsize=8, color="#666666", va="top")
ax.text(0.97, 0.04, "High Recall\nLow Precision\n" + "─" * 20 + "\nNoisy context",
        transform=ax.transAxes, fontsize=8, color="#666666", ha="right")
ax.text(0.97, 0.96, "★ IDEAL ZONE ★",
        transform=ax.transAxes, fontsize=11, color="#2e7d32", ha="right", va="top", fontweight="bold")

ax.grid(linestyle="--", alpha=0.2)

plt.tight_layout()
plt.show()

### 16f. 逐题 Recall 差异热力图

展示每个问题在不同策略下的 Recall@K 得分差异，快速定位哪些问题对策略变更敏感。

In [ ]:
# 仅当存在真实报告且至少 2 份时才构建逐题对比
has_per_question = all(
    "per_question" in report and len(report.get("per_question", [])) > 0
    for report in reports.values()
)

if has_per_question and len(reports) >= 2:
    # 使用第一个报告的 question IDs 作为行标签
    first_report = list(reports.values())[0]
    question_ids = [q["id"] for q in first_report["per_question"]]

    heatmap_data = {}
    heatmap_labels = {}
    for report_id, report in reports.items():
        top_k = report.get("top_k", TOP_K)
        label = report.get("_short_label", report_id)

        heatmap_data[label] = [
            q["core_metrics"].get(f"recall@{top_k}", 0)
            for q in report["per_question"]
        ]

    heatmap_df = pd.DataFrame(heatmap_data, index=question_ids)

    fig, ax = plt.subplots(
        figsize=(max(10, len(reports) * 2.5), max(6, len(question_ids) * 0.4)),
    )
    sns.heatmap(
        heatmap_df,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        linewidths=1,
        linecolor="white",
        ax=ax,
        cbar_kws={"label": f"Recall@{TOP_K}", "shrink": 0.8},
        annot_kws={"fontsize": 9, "fontweight": "bold"},
    )
    ax.set_title(
        f"Per-Question Recall@{TOP_K} — Strategy Comparison",
        fontweight="bold",
        fontsize=15,
        pad=15,
    )
    ax.set_ylabel("Question ID", fontsize=12)
    ax.set_xlabel("Strategy", fontsize=12)
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()

    # ── 打印逐题差异摘要 ──
    print("📊 逐题 Recall 差异分析：")
    print(f"   问题数: {len(question_ids)}")
    print(f"   策略数: {len(reports)}")

    # 找差异最大的问题
    if len(reports) >= 2:
        heatmap_df["max_diff"] = heatmap_df.max(axis=1) - heatmap_df.min(axis=1)
        sensitive = heatmap_df.nlargest(3, "max_diff")
        stable = heatmap_df.nsmallest(3, "max_diff")

        print(f"\n   🔍 对策略最敏感的 3 个问题 (Recall 波动最大)：")
        for qid, row in sensitive.iterrows():
            print(f"      {qid}: range={row['max_diff']:.2f}")

        print(f"\n   ✅ 对策略最稳定的 3 个问题 (Recall 波动最小)：")
        for qid, row in stable.iterrows():
            print(f"      {qid}: range={row['max_diff']:.2f}")
else:
    print("需要至少 2 份包含逐题数据的真实报告才能生成此热力图。")
    print("请运行 CLI 评估工具生成真实报告后回来刷新。")